# Phase 2 Integration Test (staging)

Runs the Phase 2 targeted-extraction service (`pipeline/phase2_extract`) end to end
in a disposable staging environment: creates staging GCP resources (bucket,
BigQuery dataset/table, Pub/Sub topic), deploys the Cloud Run service, publishes
realistic trigger messages, and verifies Gemini's structured output lands correctly
in BigQuery — including the `row_ids` idempotency fix from
[PR #4](https://github.com/marceloamoraes/switchb-plan/pull/4).

**Before running:** authenticate `gcloud` (`gcloud auth login` and
`gcloud auth application-default login`) in the same environment this kernel runs
in, and make sure the Vertex AI API is enabled
(`gcloud services enable aiplatform.googleapis.com`).

This notebook is self-contained and reuses the same staging bucket name and
`phase2-trigger-staging` topic as the
[Phase 1 integration test notebook](phase1_integration_test.ipynb) if you already
ran it — resource-creation cells are safe to re-run (they skip anything that
already exists) either way.

Run cells top to bottom. Step 13 (teardown) deletes the staging resources — only
run it once you're done.

## 0. Locate the repo root and install dependencies

In [ ]:
import pathlib

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "pipeline").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

assert (REPO_ROOT / "pipeline").exists(), "Could not find the repo root — run Jupyter from inside the repo."
print("Repo root:", REPO_ROOT)


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO_ROOT / "pipeline" / "phase2_extract" / "requirements.txt"),
    "google-cloud-pubsub",
])


## 1. Set variables

Edit `PROJECT_ID` before running.

In [ ]:
import os

PROJECT_ID = "your-gcp-project"  # <-- edit this
REGION = "us-central1"
STAGING_BUCKET = f"{PROJECT_ID}-switchgear-staging"
DATASET = "switchgear_staging"
TABLE = "extracted_specs"
PHASE2_TOPIC = "phase2-trigger-staging"
BQ_TABLE = f"{PROJECT_ID}.{DATASET}.{TABLE}"
TEST_PREFIX = "phase2-test/"

os.environ.update({
    "PROJECT_ID": PROJECT_ID,
    "REGION": REGION,
    "STAGING_BUCKET": STAGING_BUCKET,
    "DATASET": DATASET,
    "TABLE": TABLE,
    "PHASE2_TOPIC": PHASE2_TOPIC,
    "REPO_ROOT": str(REPO_ROOT),
})

for k in ["PROJECT_ID", "REGION", "STAGING_BUCKET", "DATASET", "TABLE", "PHASE2_TOPIC", "BQ_TABLE"]:
    print(f"{k}={locals().get(k, os.environ.get(k))}")


## 2. Create the staging bucket, BigQuery dataset/table, and Pub/Sub topic

Each `create` call is followed by `|| echo ... (already exists)` so this cell is safe to re-run.

In [ ]:
%%bash
gcloud storage buckets create "gs://$STAGING_BUCKET" --location="$REGION" \
  || echo "bucket already exists, continuing"

bq mk --dataset --location="$REGION" "$PROJECT_ID:$DATASET" \
  || echo "dataset already exists, continuing"

bq mk --table "$PROJECT_ID:$DATASET.$TABLE" "$REPO_ROOT/infra/bigquery_schema.json" \
  || echo "table already exists, continuing"

gcloud pubsub topics create "$PHASE2_TOPIC" \
  || echo "topic already exists, continuing"


## 3. Create a dedicated runtime service account (least privilege)

- `roles/storage.objectViewer` on the staging bucket, to read extracted text
- `roles/aiplatform.user` at the project level, to call Vertex AI Gemini (Vertex AI has no finer-grained resource to scope this to)
- `WRITER` access on the BigQuery dataset, to insert rows

In [ ]:
%%bash
gcloud iam service-accounts create phase2-extract-sa --display-name="Phase 2 extract runtime SA" \
  || echo "service account already exists, continuing"

gcloud storage buckets add-iam-policy-binding "gs://$STAGING_BUCKET" \
  --member="serviceAccount:phase2-extract-sa@${PROJECT_ID}.iam.gserviceaccount.com" \
  --role="roles/storage.objectViewer"

gcloud projects add-iam-policy-binding "$PROJECT_ID" \
  --member="serviceAccount:phase2-extract-sa@${PROJECT_ID}.iam.gserviceaccount.com" \
  --role="roles/aiplatform.user" --condition=None


In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET)
dataset = bq_client.get_dataset(dataset_ref)

sa_email = f"phase2-extract-sa@{PROJECT_ID}.iam.gserviceaccount.com"
already_granted = any(e.entity_id == sa_email for e in dataset.access_entries)
if not already_granted:
    dataset.access_entries = list(dataset.access_entries) + [
        bigquery.AccessEntry(role="WRITER", entity_type="serviceAccount", entity_id=sa_email)
    ]
    dataset = bq_client.update_dataset(dataset, ["access_entries"])
    print("Granted WRITER on dataset to", sa_email)
else:
    print(sa_email, "already has dataset access, skipping")


## 4. Build and deploy Phase 2 to staging

In [ ]:
%%bash
gcloud builds submit "$REPO_ROOT/pipeline/phase2_extract" --tag "gcr.io/$PROJECT_ID/phase2-extract-staging"

gcloud run deploy phase2-extract-staging \
  --image "gcr.io/$PROJECT_ID/phase2-extract-staging" \
  --region "$REGION" --no-allow-unauthenticated \
  --service-account="phase2-extract-sa@${PROJECT_ID}.iam.gserviceaccount.com" \
  --timeout=120 --memory=512Mi \
  --set-env-vars "PROJECT_ID=$PROJECT_ID,REGION=$REGION,BQ_TABLE=${PROJECT_ID}.${DATASET}.${TABLE}"


## 5. Wire up the push subscription

In [ ]:
%%bash
gcloud iam service-accounts create run-invoker --display-name="Pub/Sub push invoker" \
  || echo "service account already exists, continuing"

gcloud run services add-iam-policy-binding phase2-extract-staging \
  --region="$REGION" \
  --member="serviceAccount:run-invoker@${PROJECT_ID}.iam.gserviceaccount.com" \
  --role="roles/run.invoker"

PHASE2_URL=$(gcloud run services describe phase2-extract-staging --region="$REGION" --format='value(status.url)')

gcloud pubsub subscriptions create phase2-sub-staging \
  --topic="$PHASE2_TOPIC" --push-endpoint="$PHASE2_URL" \
  --push-auth-service-account="run-invoker@${PROJECT_ID}.iam.gserviceaccount.com" \
  || echo "subscription already exists, continuing"


## 6. Prepare a test corpus

Two synthetic switchgear spec snippets, written locally, so this notebook is
runnable without needing real documents. For a real data-quality signal, also copy
5-10 real `extracted/*.txt` files from the Phase 1 staging bucket into
`phase2-test-data/` before running the next cell.

In [ ]:
TEST_DATA_DIR = pathlib.Path("phase2-test-data")
TEST_DATA_DIR.mkdir(exist_ok=True)

samples = {
    "sample_switchboard_1.txt": """Switchboard Assembly - Model SB-4000
Vendor: Eaton Corporation
Component: SB-4000-3000A
Product: 3000A Main Switchboard, Type 1 Enclosure, NEMA 1
Ampacity: 3000A
Voltage: 480/277V
Enclosure type: NEMA1
Options: Digital Power Meter, Ground Fault Protection
Circuit Breaker Details: LCB 1 Right: 3000A Electronic LSIG (100kA @ 480V); LCB 2 Left: 1200A Electronic LSI (65kA @ 480V)
Total Price: $124,500.00
""",
    "sample_switchboard_2.txt": """Switch board unit - outdoor rated
Vendor: Square D / Schneider Electric
Component: QED-2-800
Product: 800A Distribution Switchboard, NEMA3R outdoor enclosure
Ampacity: 800A
Voltage: 208Y/120V
Enclosure type: NEMA3R
Options: Thermostat, Internal heaters, Automatic shutters
Circuit Breaker Details: Main: 800AS/800AT Type PJ Ammeter Trip Unit; Feeders: 225AS/200AT Type LL, 30AT Type BJ
Total Price: $58,900.00
""",
}

for name, content in samples.items():
    (TEST_DATA_DIR / name).write_text(content)

test_files = sorted(TEST_DATA_DIR.glob("*.txt"))
print(f"{len(test_files)} test file(s) in {TEST_DATA_DIR.resolve()}:")
for p in test_files:
    print(" ", p.name)


## 7. Upload the test corpus

In [ ]:
from google.cloud import storage

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(STAGING_BUCKET)

text_objects = []
for path in test_files:
    blob_name = f"{TEST_PREFIX}{path.name}"
    bucket.blob(blob_name).upload_from_filename(str(path), content_type="text/plain")
    text_objects.append(blob_name)
    print("Uploaded", blob_name)


## 8. Publish trigger messages

Publishes one Pub/Sub message per uploaded file, in the exact JSON shape `pipeline/phase2_extract/main.py` expects: `{"bucket", "source_object", "text_object"}`.

In [ ]:
import json as _json
from google.cloud import pubsub_v1

publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(PROJECT_ID, PHASE2_TOPIC)

published_source_objects = []
for text_object in text_objects:
    source_object = text_object  # using the .txt path itself as a stand-in source_object for this test
    payload = {"bucket": STAGING_BUCKET, "source_object": source_object, "text_object": text_object}
    publisher.publish(topic_path, _json.dumps(payload).encode("utf-8")).result()
    published_source_objects.append(source_object)
    print("Published:", payload)


## 9. Check the logs

A bounded, non-blocking read — re-run this cell to refresh.

In [ ]:
%%bash
gcloud logging read \
  'resource.type="cloud_run_revision" AND resource.labels.service_name="phase2-extract-staging"' \
  --project="$PROJECT_ID" --limit=50 --freshness=10m \
  --format="value(timestamp, textPayload)"


## 10. Verify the BigQuery rows

Confirms Gemini's structured output landed, and spot-checks that the required fields aren't empty.

In [ ]:
rows = list(bq_client.query(f"""
    SELECT *
    FROM `{BQ_TABLE}`
    WHERE source_uri IN UNNEST(@source_uris)
    ORDER BY processed_at DESC
""", job_config=bigquery.QueryJobConfig(query_parameters=[
    bigquery.ArrayQueryParameter(
        "source_uris", "STRING",
        [f"gs://{STAGING_BUCKET}/{s}" for s in published_source_objects],
    )
])))

required_fields = [
    "component", "Product", "Vendor", "Ampacity", "Voltage",
    "Enclosure type", "Options", "Circuit Breaker Details", "Total Price",
]
for row in rows:
    row = dict(row)
    missing = [f for f in required_fields if not row.get(f)]
    print(row["source_uri"], "- missing/empty fields:", missing or "none")
    print(" ", {k: row[k] for k in required_fields})


## 11. Measure cost per document

Calls Gemini directly (bypassing Cloud Run) against the same local test files, to
capture token usage without needing to instrument the production service. Edit the
per-million-token prices below from the current
[Vertex AI Gemini pricing page](https://cloud.google.com/vertex-ai/generative-ai/pricing)
before trusting the dollar figures — they're placeholders.

In [ ]:
import sys

sys.path.insert(0, str(REPO_ROOT / "pipeline" / "phase2_extract"))
from schema import RESPONSE_SCHEMA, build_prompt
from google import genai

GEMINI_MODEL = "gemini-2.5-flash"
INPUT_PRICE_PER_MILLION_TOKENS = 0.0   # <-- edit from the current pricing page
OUTPUT_PRICE_PER_MILLION_TOKENS = 0.0  # <-- edit from the current pricing page

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

total_cost = 0.0
for path in test_files:
    text = path.read_text()
    response = genai_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=build_prompt(text),
        config={"response_mime_type": "application/json", "response_schema": RESPONSE_SCHEMA},
    )
    usage = response.usage_metadata
    cost = (
        usage.prompt_token_count / 1_000_000 * INPUT_PRICE_PER_MILLION_TOKENS
        + usage.candidates_token_count / 1_000_000 * OUTPUT_PRICE_PER_MILLION_TOKENS
    )
    total_cost += cost
    print(f"{path.name}: {usage.prompt_token_count} in / {usage.candidates_token_count} out tokens, ~${cost:.6f}")

print(f"\nAverage cost per document: ~${total_cost / len(test_files):.6f}")
print(f"Extrapolated to 130,000 documents at this rate: ~${total_cost / len(test_files) * 130_000:,.2f}")
print("(only meaningful once matched-doc rate and real pricing are filled in above)")


## 12. Verify the idempotency fix ([PR #4](https://github.com/marceloamoraes/switchb-plan/pull/4))

Publishes the same payload twice for one object, then confirms BigQuery ends up with exactly one row for it rather than two.

In [ ]:
dup_payload = {
    "bucket": STAGING_BUCKET,
    "source_object": text_objects[0],
    "text_object": text_objects[0],
}
for _ in range(2):
    publisher.publish(topic_path, _json.dumps(dup_payload).encode("utf-8")).result()
    print("Published duplicate:", dup_payload)


Wait ~15-30 seconds for both deliveries to be processed, then check the count:

In [ ]:
import time

time.sleep(20)

count = list(bq_client.query(f"""
    SELECT COUNT(*) AS n
    FROM `{BQ_TABLE}`
    WHERE source_uri = @source_uri
""", job_config=bigquery.QueryJobConfig(query_parameters=[
    bigquery.ScalarQueryParameter("source_uri", "STRING", f"gs://{STAGING_BUCKET}/{dup_payload['source_object']}")
])))[0]["n"]

print("Row count for duplicated source_uri:", count)
assert count == 1, f"Expected exactly 1 row (dedup via row_ids), found {count}"
print("Idempotency check passed.")


## 13. Teardown

Deletes every staging resource created above. Only run this once you're done testing. Leaves the shared `switchgear_staging` BigQuery dataset and staging bucket in place by default since the Phase 1 notebook may still be using them — uncomment the last two lines to remove those too.

In [ ]:
%%bash
gcloud run services delete phase2-extract-staging --region="$REGION" -q
gcloud pubsub subscriptions delete phase2-sub-staging -q
gsutil -m rm -r "gs://$STAGING_BUCKET/$TEST_PREFIX" 2>/dev/null || true

# bq rm -r -f -d "$PROJECT_ID:$DATASET"
# gsutil -m rm -r "gs://$STAGING_BUCKET"
